# Crypto Order Book Depth Chart Analysis & Visualization


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Crypto Order Book Depth Chart Analysis & Visualization

This notebook demonstrates how to read and analyze cryptocurrency order book depth charts like a professional trader. Based on practical order book analysis techniques, we'll build real order book visualizations, compute liquidity metrics, and identify whale activity patterns.

**Key Topics:**
- Bid/Ask spread analysis
- Liquidity wall detection
- Volume distribution patterns
- Trading signal recognition

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

np.random.seed(42)

# Generate realistic synthetic order book data
# Order books contain bid side (buy orders) and ask side (sell orders)

current_price = 45000  # Bitcoin example

# BID SIDE: prices below current, volume increases as you go deeper
bid_prices = np.array([current_price - i*10 for i in range(1, 51)])[::-1]
bid_volumes = np.random.gamma(shape=2, scale=0.5, size=50)
bid_volumes = bid_volumes * (1 + 0.003 * np.arange(50))  # Deeper orders larger

# ASK SIDE: prices above current, volume increases as you go deeper
ask_prices = np.array([current_price + i*10 for i in range(1, 51)])
ask_volumes = np.random.gamma(shape=2, scale=0.5, size=50)
ask_volumes = ask_volumes * (1 + 0.003 * np.arange(50))

# Create DataFrames
bids_df = pd.DataFrame({
    'price': bid_prices,
    'volume': bid_volumes,
    'cumulative_volume': np.cumsum(bid_volumes[::-1])[::-1]
})

asks_df = pd.DataFrame({
    'price': ask_prices,
    'volume': ask_volumes,
    'cumulative_volume': np.cumsum(ask_volumes)
})

print("Top 5 BID orders (highest prices, most likely to fill):")
print(bids_df.head())
print("\nTop 5 ASK orders (lowest prices, most likely to fill):")
print(asks_df.head())

In [ ]:
# Calculate key liquidity metrics

best_bid = bids_df['price'].max()
best_ask = asks_df['price'].min()
bid_ask_spread = best_ask - best_bid
spread_percent = (bid_ask_spread / current_price) * 100

print(f"Current Price: ${current_price}")
print(f"Best Bid (highest buy order): ${best_bid}")
print(f"Best Ask (lowest sell order): ${best_ask}")
print(f"Bid-Ask Spread: ${bid_ask_spread}")
print(f"Spread %: {spread_percent:.3f}%")
print()

# Liquidity at different depth levels
for depth in [100000, 500000, 1000000]:
    bid_vol = bids_df[bids_df['cumulative_volume'] <= depth]['volume'].sum()
    ask_vol = asks_df[asks_df['cumulative_volume'] <= depth]['volume'].sum()
    print(f"Liquidity within ${depth:,}:")
    print(f"  Bid side: {bid_vol:.2f} BTC | Ask side: {ask_vol:.2f} BTC")

In [ ]:
# Visualize the order book depth chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# LEFT: Classic depth chart (mirrored view)
ax1.fill_between(bids_df['cumulative_volume'], bids_df['price'], 
                  best_bid, alpha=0.3, color='green', label='Bids')
ax1.fill_between(asks_df['cumulative_volume'], asks_df['price'], 
                  best_ask, alpha=0.3, color='red', label='Asks')
ax1.axhline(y=current_price, color='blue', linestyle='--', linewidth=2, label='Current Price')
ax1.set_xlabel('Cumulative Volume (BTC)', fontsize=11)
ax1.set_ylabel('Price (USD)', fontsize=11)
ax1.set_title('Order Book Depth Chart', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, bids_df['cumulative_volume'].max() * 1.1)

# RIGHT: Side-by-side view (bids on left, asks on right)
ax2.barh(bids_df['price'], -bids_df['volume'], color='green', alpha=0.6, label='Bids')
ax2.barh(asks_df['price'], asks_df['volume'], color='red', alpha=0.6, label='Asks')
ax2.axvline(x=0, color='black', linewidth=1)
ax2.axhline(y=current_price, color='blue', linestyle='--', linewidth=2, label='Current Price')
ax2.set_xlabel('Volume (BTC)', fontsize=11)
ax2.set_ylabel('Price (USD)', fontsize=11)
ax2.set_title('Order Book Ladder View', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# WHALE DETECTION: Identify unusually large orders (liquidity walls)

# Calculate statistics for order volumes
bid_vol_mean = bids_df['volume'].mean()
bid_vol_std = bids_df['volume'].std()
ask_vol_mean = asks_df['volume'].mean()
ask_vol_std = asks_df['volume'].std()

# Whale threshold: orders > mean + 1.5*std
bid_whale_threshold = bid_vol_mean + 1.5 * bid_vol_std
ask_whale_threshold = ask_vol_mean + 1.5 * ask_vol_std

bid_whales = bids_df[bids_df['volume'] > bid_whale_threshold].copy()
ask_whales = asks_df[asks_df['volume'] > ask_whale_threshold].copy()

print(f"BID WHALE WALLS (volume > {bid_whale_threshold:.2f} BTC):")
print(bid_whales[['price', 'volume']].to_string())
print(f"\nASK WHALE WALLS (volume > {ask_whale_threshold:.2f} BTC):")
print(ask_whales[['price', 'volume']].to_string())
print(f"\nInterpretation:")
print(f"- Bid whales act as SUPPORT: large buy orders at lower prices can slow a drop")
print(f"- Ask whales act as RESISTANCE: large sell orders at higher prices can limit upside")

In [ ]:
# Analyze liquidity distribution and spot trading signals

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: Cumulative volume by depth
ax = axes[0, 0]
ax.plot(bids_df['price'], bids_df['cumulative_volume'], 'g-', linewidth=2, label='Bid Cumulative')
ax.plot(asks_df['price'], asks_df['cumulative_volume'], 'r-', linewidth=2, label='Ask Cumulative')
ax.axvline(x=current_price, color='blue', linestyle='--', linewidth=2, alpha=0.7)
ax.set_xlabel('Price (USD)', fontsize=10)
ax.set_ylabel('Cumulative Volume (BTC)', fontsize=10)
ax.set_title('Cumulative Liquidity by Price', fontsize=11, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Top-right: Volume distribution (histogram)
ax = axes[0, 1]
ax.hist(bids_df['volume'], bins=20, alpha=0.6, color='green', label='Bid Volume')
ax.hist(asks_df['volume'], bins=20, alpha=0.6, color='red', label='Ask Volume')
ax.axvline(bid_vol_mean, color='green', linestyle='--', linewidth=2, label=f'Bid Mean')
ax.axvline(ask_vol_mean, color='red', linestyle='--', linewidth=2, label=f'Ask Mean')
ax.set_xlabel('Order Volume (BTC)', fontsize=10)
ax.set_ylabel('Frequency', fontsize=10)
ax.set_title('Order Volume Distribution', fontsize=11, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Bottom-left: Bid-Ask Imbalance (buy vs sell pressure)
ax = axes[1, 0]
volume_ladder = 30
bid_imbalance = bids_df.head(volume_ladder)['volume'].values - asks_df.head(volume_ladder)['volume'].values
colors = ['green' if x > 0 else 'red' for x in bid_imbalance]
ax.bar(range(volume_ladder), bid_imbalance, color=colors, alpha=0.6)
ax.axhline(y=0, color='black', linewidth=1)
ax.set_xlabel('Order Depth Rank', fontsize=10)
ax.set_ylabel('Volume Imbalance (Bid - Ask)', fontsize=10)
ax.set_title('Bid-Ask Volume Imbalance (First 30 Levels)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Bottom-right: Key metrics summary
ax = axes[1, 1]
ax.axis('off')
summary_text = f"""
KEY ORDER BOOK SIGNALS:

1. SPREAD ANALYSIS
   • Spread: ${bid_ask_spread:.0f} ({spread_percent:.3f}%)
   • Tight spread = high liquidity & low friction
   • Wide spread = illiquid or volatile conditions

2. LIQUIDITY WALLS
   • Bid walls = support zones (buying pressure)
   • Ask walls = resistance zones (selling pressure)
   • {len(bid_whales)} whale bids | {len(ask_whales)} whale asks detected

3. DEPTH ANALYSIS
   • Compare cumulative volume at price levels
   • Steep curve = concentrated liquidity
   • Flat curve = distributed orders (harder to move price)

4. IMBALANCE
   • More volume on bids = bullish pressure
   • More volume on asks = bearish pressure
   • Can signal upcoming move direction
"""
ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
        fontsize=10, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.show()

## Trading Signals from Order Book Analysis

### 1. **Whale Detection**
Identify unusually large orders that can move the market. Whales create "support" (bids) and "resistance" (asks).

### 2. **Spread Tightness**
A narrow bid-ask spread indicates high liquidity and confidence. A widening spread may signal uncertainty or low liquidity.

### 3. **Liquidity Depth**
Check how much volume is available at different price levels. Shallow order books (little volume deep down) can be gamed by large trades.

### 4. **Volume Imbalance**
When bids significantly outweigh asks near market price, buyers are more aggressive (bullish signal). The reverse suggests selling pressure.

### 5. **Walls & Levels**
Large orders at round numbers or technical price levels often indicate institutional interest and can act as support/resistance.

## Practical Tips
- **Check multiple exchanges:** Order book structure varies; illiquid venues show wider spreads.
- **Monitor dynamic changes:** Walls disappearing = reduced support/resistance.
- **Combine with price action:** Order book alone doesn't predict direction; use with candlestick patterns.
- **Watch during volatility:** Whales often place large orders during uncertain market conditions.

---

## Reference

[https://protraderdaily.com](https://protraderdaily.com/analysis/how-to-read-crypto-order-book-depth-charts-for-trading)
